In [1]:
## Model Training

#### 1.1 Import Data and Required Packages
##### Importing Pandas, Numpy, Matplotlib, Seaborn and Warings Library.

In [2]:
# Basic Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing & Data Splitting
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Regression Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

# Evaluation Metrics
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

#### Import the CSV Data as Pandas DataFrame

In [3]:
import pandas as pd
import numpy as np
df = pd.read_csv('data/car_data.csv')
df.head()

,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


#### Show Top 5 Records

In [4]:
df.head()

,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


#### Preparing X and Y variables

In [5]:
# 1. Feature Engineering: Create Car_Age and drop Year
df['Car_Age'] = 2026 - df['Year']
df.drop(columns=['Year'], inplace=True)

# 2. Separate Features (X) and Target (y)
X = df.drop(columns=['Selling_Price', 'Car_Name'])
y = df['Selling_Price']

X.head()

,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner,Car_Age
0,5.59,27000,Petrol,Dealer,Manual,0,12
1,9.54,43000,Diesel,Dealer,Manual,0,13
2,9.85,6900,Petrol,Dealer,Manual,0,9
3,4.15,5200,Petrol,Dealer,Manual,0,15
4,6.87,42450,Diesel,Dealer,Manual,0,12


In [6]:
X.head()

,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner,Car_Age
0,5.59,27000,Petrol,Dealer,Manual,0,12
1,9.54,43000,Diesel,Dealer,Manual,0,13
2,9.85,6900,Petrol,Dealer,Manual,0,9
3,4.15,5200,Petrol,Dealer,Manual,0,15
4,6.87,42450,Diesel,Dealer,Manual,0,12


In [7]:
print("Categories in 'Fuel_Type' variable:    ", end=" ")
print(df['Fuel_Type'].unique())

print("Categories in 'Seller_Type' variable:  ", end=" ")
print(df['Seller_Type'].unique())

print("Categories in 'Transmission' variable: ", end=" ")
print(df['Transmission'].unique())
print("Categories in 'Owner' variable:        ", end=" ")
print(df['Owner'].unique())

Categories in 'Fuel_Type' variable:     <ArrowStringArray>
['Petrol', 'Diesel', 'CNG']
Length: 3, dtype: str
Categories in 'Seller_Type' variable:   <ArrowStringArray>
['Dealer', 'Individual']
Length: 2, dtype: str
Categories in 'Transmission' variable:  <ArrowStringArray>
['Manual', 'Automatic']
Length: 2, dtype: str
Categories in 'Owner' variable:         [0 1 3]


In [8]:
y = df['Selling_Price']

In [9]:
y

0       3.35
1       4.75
2       7.25
3       2.85
4       4.60
       ...  
296     9.50
297     4.00
298     3.35
299    11.50
300     5.30
Name: Selling_Price, Length: 301, dtype: float64

In [10]:
# 1. Reload clean data
df = pd.read_csv('data/car_data.csv')

# 2. Feature Engineering
df['Car_Age'] = 2026 - df['Year']

# 3. Define Features (X) and Target (y)
X = df.drop(columns=['Selling_Price', 'Car_Name', 'Year'])
y = df['Selling_Price']

# 4. Identify Categorical & Numerical Features
num_features = X.select_dtypes(include=[np.number]).columns
cat_features = X.select_dtypes(include=['object', 'string']).columns

# 5. Define Transformers
numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(sparse_output=False, drop='first')

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features),        
    ]
)

# 6. Fit and Transform
X = preprocessor.fit_transform(X)

print("Preprocessed X shape:", X.shape)

Preprocessed X shape: (301, 8)


In [11]:
X = preprocessor.fit_transform(X)

ValueError: Specifying the columns using strings is only supported for dataframes.

In [ ]:
X.shape

In [ ]:
# Separate dataset into train and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape

#### Create an Evaluate Function to give all metrics after model Training

In [ ]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest Regressor": RandomForestRegressor(random_state=42),
    "XGBRegressor": XGBRegressor(random_state=42), 
    "CatBoosting Regressor": CatBoostRegressor(verbose=False, random_state=42),
    "AdaBoost Regressor": AdaBoostRegressor(random_state=42)
}

model_list = []
r2_list = []

for model_name, model in models.items():
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    model_train_mae, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)
    
    model_list.append(model_name)
    r2_list.append(model_test_r2)

In [ ]:
### Results

In [ ]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"], ascending=False)

## Linear Regression

In [ ]:
lin_model = LinearRegression(fit_intercept=True)
lin_model = lin_model.fit(X_train, y_train)
y_pred = lin_model.predict(X_test)
score = r2_score(y_test, y_pred) * 100
print("Accuracy of the model is %.2f%%" % score)

## Plot y_pred and y_test

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.7, color='blue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2) # Ideal fit reference line

plt.xlabel('Actual Selling Price')
plt.ylabel('Predicted Selling Price')
plt.title('Actual vs Predicted Car Prices')
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.regplot(x=y_test, y=y_pred, ci=None, color='red', scatter_kws={'alpha': 0.6})

plt.xlabel('Actual Selling Price')
plt.ylabel('Predicted Selling Price')
plt.title('Actual vs. Predicted Car Selling Prices')
plt.grid(True)
plt.show()

#### Difference between Actual and Predicted Values

In [ ]:
pred_df = pd.DataFrame({
    'Actual Value': y_test,
    'Predicted Value': y_pred,
    'Difference': y_test - y_pred
})

pred_df.head(10) # Display the top 10 predictions cleanly

In [14]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Selling_Price'])
y = df['Selling_Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
import joblib

# 1. Transform raw training data using preprocessor
X_train_transformed = preprocessor.fit_transform(X_train)

# 2. Train best performing model on the transformed data
best_model = RandomForestRegressor(random_state=42)
best_model.fit(X_train_transformed, y_train)

# 3. Save preprocessor and best model
joblib.dump(preprocessor, 'car_preprocessor.pkl')
joblib.dump(best_model, 'car_price_model.pkl')

print("Preprocessor and Model successfully saved to disk!")

Preprocessor and Model successfully saved to disk!


In [18]:
import joblib
import pandas as pd

# Load saved preprocessor and model
loaded_preprocessor = joblib.load('car_preprocessor.pkl')
loaded_model = joblib.load('car_price_model.pkl')

# Test prediction on raw input
new_car = pd.DataFrame([{
    'Present_Price': 5.59,
    'Kms_Driven': 27000,
    'Owner': 0,
    'Car_Age': 8,
    'Fuel_Type': 'Petrol',
    'Seller_Type': 'Dealer',
    'Transmission': 'Manual'
}])

new_car_processed = loaded_preprocessor.transform(new_car)
predicted_price = loaded_model.predict(new_car_processed)[0]

print(f"Predicted Selling Price: ₹{predicted_price:.2f} Lakhs")

Predicted Selling Price: ₹4.15 Lakhs
